In [148]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib tk
# %matplotlib inline


In [149]:
from scipy.signal.windows import tukey
# ---- LOAD GRAV WAVE DATA AND TAPER
from helpers import *

time, strain = get_sample_data()
window_function = tukey(M=len(time), alpha=0.1, sym=True)

strain_tapered = window_function * strain

t = time
dt = t[1] - t[0]
d = strain_tapered

# ---- OTHER FUNCTION
# def periodic_fun(x):
#     return (
#         np.sin(x)
#         + 0.3*np.sin(3*x + 0.5)
#         + 0.2*np.cos(5*x - 1.2)
#         + 0.1*np.sin(7*x**0.5)  # adds some irregularity, still periodic
#     )

# Example usage
# t = np.linspace(0, 10*np.pi, 300)
# dt = t[1] - t[0]
# d = periodic_fun(t)


In [150]:
_=plt.figure()
plt.plot(t, d)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Data $d$ (periodic)")

Text(0.5, 1.0, 'Data $d$ (periodic)')

In [285]:
d_tilde = np.fft.fft(d, norm="ortho")  # norm=backward => forward transform unscaled
k_lengths = np.fft.fftfreq(n=len(d), d=dt)
k_lengths_unique = np.fft.rfftfreq(n=len(d), d=dt)
print("k lengths: ", k_lengths)
print("k lengths unique: ", k_lengths_unique)

k lengths:  [ 0.          0.50006104  1.0001221  ... -1.5001831  -1.0001221
 -0.50006104]
k lengths unique:  [0.0000000e+00 5.0006104e-01 1.0001221e+00 ... 2.0467498e+03
 2.0472499e+03 2.0477500e+03]


In [152]:
_=plt.figure()
plt.plot(k_lengths, d_tilde, "-")

/Users/iason/PycharmProjects/stability-of-submoons/.venv/lib/python3.12/site-packages/matplotlib/cbook/__init__.py:1345: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)


In [299]:
from typing import Literal

def data_model(amplitude_spectrum, custom_xi=None, mode:Literal["hermitian","even"]="hermitian", custom_norm_for_your_convenience=1):
    # spits out a realization given a power spectrum
    # power spectrum should have length len(data), i.e. be already distributed
    if custom_xi is None:
        if mode == "even":
            xi = np.random.standard_normal(len(amplitude_spectrum))
        elif mode == "hermitian":
            xi = 1/np.sqrt(2) *(np.random.normal(size=len(amplitude_spectrum)) + 1j * np.random.normal(size=len(amplitude_spectrum)))
        else:
            raise ValueError("Unknown mode")

    else:
        xi = custom_xi
    return np.fft.ifft(custom_norm_for_your_convenience * amplitude_spectrum * xi, norm="ortho").real


def expand_rfft(f_unique, N):
    # work with unique k's and broadcast to full k's using this function.
    return np.concatenate([f_unique, f_unique[-2:0:-1].conj()]) if N % 2 == 0 else np.concatenate([f_unique, f_unique[-1:0:-1].conj()])


def mean_prior_power_spectrum(k, p):
    # p = params, k fourier modes
    # assumes k[0] = 0 and np ordering of k

    slope = p[0]
    amplitude = p[1]

    tmp = np.abs(k.copy())  # negative modes are just the positive ones mirrored. If you remove abs you will get an error for slope =-1 for example, makes sense.
    tmp[0] = 1  # mask zeromode

    tmp = tmp**slope

    sorter = np.argsort(k)

    tmp = tmp / (np.trapz(tmp[sorter][1:], k[sorter][1:]))
    tmp = amplitude * tmp
    tmp[0] = 1e-30  # fix zeromode

    if not np.all(tmp >=0 ):

        raise ValueError("Power spectrum cannot be negative, p_s(k) = ", tmp)

    return expand_rfft(tmp, len(d))

In [169]:
import numpy as np

def solve_s_components(tilde_d, s0, sigma, tol=1e-9, maxiter=50):
    # arrays
    td = np.abs(np.asarray(tilde_d))
    s0 = np.asarray(s0)
    # initial guess
    s = np.maximum(s0, np.sqrt(sigma * td))
    eps = 1e-12

    for _ in range(maxiter):
        f = s**4 - s0 * s**3 - (sigma**2) * (td**2)
        df = 4*s**3 - 3*s0 * s**2

        # protect tiny derivative
        small_df = np.abs(df) < 1e-12
        # Newton step
        step = np.empty_like(s)
        step[~small_df] = f[~small_df] / df[~small_df]
        step[small_df] = f[small_df] / (1e-12 + df[small_df])  # tiny-deriv fallback

        s_new = s - step
        # keep positive
        s_new = np.maximum(s_new, eps)

        # convergence check
        relchg = np.max(np.abs((s_new - s) / np.maximum(s, eps)))
        s = s_new
        if relchg < tol:
            break

    # final safety: if any f(s) still large, do simple bracketed bisection per component
    # bad = np.where(np.abs(s**4 - s0*s**3 - (sigma**2)*(td**2)) > 1e-6)[0]
    # for i in bad:
    #     lo = eps
    #     hi = max(s0.flat[i], np.sqrt(sigma * td.flat[i])) * 20.0
    #     for _ in range(80):
    #         mid = 0.5*(lo+hi)
    #         fm = mid**4 - s0.flat[i]*mid**3 - (sigma**2)*(td.flat[i]**2)
    #         if fm == 0:
    #             lo = hi = mid; break
    #         flo = lo**4 - s0.flat[i]*lo**3 - (sigma**2)*(td.flat[i]**2)
    #         # sign check
    #         if np.sign(flo) == np.sign(fm):
    #             lo = mid
    #         else:
    #             hi = mid
    #         if (hi-lo)/hi < 1e-9: break
    #     s.flat[i] = 0.5*(lo+hi)

    return s


In [170]:
s0 = np.sqrt(mean_prior_power_spectrum(k_lengths, (-4, 2e4)))
sig_amp_spec = 1e-3

data_samples = []
for _ in range(10):
    sl = data_model(s0)
    data_samples.append(sl)


amp_spec_samples = []
for _ in range(3):
    # log_s0 = np.log(s0)
    # log_sl = log_s0 + sig_amp_spec * np.random.standard_normal(len(s0))
    # amp_spec_samples.append(np.exp(log_sl))
    sl = s0 + sig_amp_spec * np.random.standard_normal(len(s0))
    amp_spec_samples.append(sl)


fig, axs = plt.subplots(1,2)


for sl in amp_spec_samples:
    axs[0].plot(k_lengths[1:], (sl**2)[1:], "-", markersize=3, alpha=0.1, color="black")

axs[0].plot(k_lengths[1:], (s0**2)[1:], ".", markersize=3, label="Mean prior power spectrum")
axs[0].loglog()

for sl in data_samples:
    axs[1].plot(t, sl, alpha=0.1, color="black")


axs[1].plot(t, data_samples[0], label="Single data realization")
axs[0].legend()
axs[1].legend()
axs[0].set_title("Prior power spectra")
axs[1].set_title("Data realizations from mean prior power spectrum")




Text(0.5, 1.0, 'Data realizations from mean prior power spectrum')

In [145]:
posterior_amplitude_spectrum = solve_s_components(tilde_d=d_tilde, s0=s0, sigma=sig_amp_spec)

In [146]:
posterior_data_samples = []
for _ in range(10):
    sl = data_model(posterior_amplitude_spectrum)
    posterior_data_samples.append(sl)


In [147]:

fig, axs = plt.subplots(1,2, figsize=(10,4))

axs[0].plot(k_lengths, posterior_amplitude_spectrum**2, ".", markersize=3, label="posterior power spectrum")
axs[0].plot(k_lengths[1:], (s0**2)[1:], ".", markersize=3, label="prior power spectrum", color="orange")
axs[0].loglog()

for sl in posterior_data_samples:
    axs[1].plot(t, sl, alpha=0.1)

mean_data_posterior = np.mean(posterior_data_samples, axis=0)
axs[1].plot(t, mean_data_posterior, color="blue", label="Mean data from posterior")
axs[1].plot(t, d, "r-", label="Actual data")

axs[1].legend(fontsize=8)
axs[0].legend(fontsize=8)

axs[0].set_title("power spectrum")
axs[1].set_title("data space")

/Users/iason/PycharmProjects/stability-of-submoons/.venv/lib/python3.12/site-packages/matplotlib/cbook/__init__.py:1345: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)


Text(0.5, 1.0, 'data space')

In [97]:
s_hat_inverse_posterior = np.diag(1/posterior_amplitude_spectrum)
posterior_xi = s_hat_inverse_posterior @ d_tilde

In [98]:
posterior_data = data_model(posterior_amplitude_spectrum, custom_xi=posterior_xi)

In [99]:
_ = plt.figure()
plt.plot(k_lengths, posterior_xi, ".", markersize=3)
plt.title(r"Posterior $\xi$ for perfect data fit")

Text(0.5, 1.0, 'Posterior $\\xi$ for perfect data fit')

In [100]:
_ = plt.figure()
plt.plot(t, posterior_data)
plt.plot(t, d, "r.", markersize=3)
plt.title("Posterior vs real data")

Text(0.5, 1.0, 'Posterior vs real data')

Does the found power spectrum look like the |fourier data|^2?

In [21]:
ps_sample = np.abs(d_tilde)**2

In [110]:
_ = plt.figure()
plt.plot(k_lengths, np.sqrt(ps_sample), "r.", markersize=2, label="Fourier transformed data")
plt.plot(k_lengths, posterior_amplitude_spectrum, "b.", markersize=2, label="Posterior amplitude spectrum")
plt.loglog()
plt.legend()

Now just check what happens if you input that xi into the wigner function?

In [ ]:
real_space_posterior_xi = np.fft.fft(posterior_xi, norm="ortho")

In [ ]:
_ = plt.figure()
plt.plot(t, real_space_posterior_xi, "b.", markersize=2)

In [ ]:
import nifty8 as ift
xi_field = ift.Field(domain=ift.DomainTuple.make(ift.RGSpace(shape=(len(t)), distances=dt),), val=real_space_posterior_xi)

In [ ]:
# stress_mat, time_dual, freq_dual = Stress(xi_field)

In [ ]:
# visualize_stress(stress_mat, rows=freq_dual, cols=time_dual+t[0])

# Correlated prior: Tridiagonal precision matrix

We want to do the whole analysis again but this time enforce smoothness on the power spectrum

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla  # for sampling!

def make_tridiag_Q(N, sigma, kappa=0.4):
    """

    :param N:           The dimension of the matrix
    :param sigma:       The variance of each random variate (assumed constant over all bins)
    :param kappa:       The correlation coefficient between nearest neighbours! Should be at most 1/2 sigma for positiveness!
    :return:
    """
    if np.array([sigma]).flatten().shape != (1,):
        main = 1/sigma**2
    else:
        main = 1/sigma**2 * np.ones(N)

    off  = -kappa * np.ones(N-1)
    Q = sp.diags([off, main, off], offsets=[-1,0,1], format="csc")
    return Q

S_inv = make_tridiag_Q(len(d), sigma=1e1, kappa=1e2)

print(f"Q norm: {np.linalg.norm(S_inv.data):.2e}")
# Sparse LU factorization
lu = spla.splu(S_inv)


In [ ]:
prior_correlated_amp_spec_samples = []

for _ in range(1):
    xi = np.random.randn(len(s0))
    sl = lu.solve(xi) + s0
    prior_correlated_amp_spec_samples.append(sl)

In [ ]:
data_samples = []
for _ in range(10):
    sl = data_model(s0)
    data_samples.append(sl)


fig2, axs2 = plt.subplots(1,2)


for sl in prior_correlated_amp_spec_samples:
    axs2[0].plot(k_lengths[1:], (sl**2)[1:], "-", markersize=3, alpha=0.3, color="black")

axs2[0].plot(k_lengths[1:], (s0**2)[1:], ".", markersize=3, label="Mean prior power spectrum")
axs2[0].loglog()

for sl in data_samples:
    axs2[1].plot(t, sl, alpha=0.1, color="black")


axs2[1].plot(t, data_samples[0], label="Single data realization")
axs2[0].legend()
axs2[1].legend()
axs2[0].set_title("Prior power spectra")
axs2[1].set_title("Data realizations from mean prior power spectrum")

plt.show()


In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.sparse import issparse

def residual_function(s, Q, s0, d_tilde):
    """
    Compute residual: F_k(s) = s_k^3 * (Q(s - s0))_k - |d_tilde_k|^2

    Note: d_tilde can be complex; we use |d_tilde|^2 = d_tilde * conj(d_tilde)

    Parameters:
    -----------
    s : array, shape (n,)
        Current solution vector (real-valued)
    Q : sparse or dense matrix, shape (n, n)
        System matrix (can be sparse CSC format)
    s0 : array, shape (n,)
        Reference vector (real-valued)
    d_tilde : array, shape (n,)
        Target vector (can be complex)

    Returns:
    --------
    residual : array, shape (n,)
        Real-valued residual
    """
    diff = s - s0
    if issparse(Q):
        Q_diff = Q.dot(diff)
    else:
        Q_diff = Q @ diff

    # |d_tilde|^2 = d_tilde * conj(d_tilde) = real part automatically
    d_tilde_mag_sq = np.abs(d_tilde)**2

    residual = s**3 * Q_diff - d_tilde_mag_sq
    return residual


def jacobian_function(s, Q, s0, d_tilde):
    """
    Compute Jacobian: J_kj = 3*s_k^2*delta_kj*(Q(s-s0))_k + s_k^3*Q_kj

    Parameters:
    -----------
    s : array, shape (n,)
        Current solution vector
    Q : sparse or dense matrix, shape (n, n)
        System matrix
    s0 : array, shape (n,)
        Reference vector
    d_tilde : array, shape (n,)
        Target vector (unused but kept for consistency)

    Returns:
    --------
    J : array or sparse matrix, shape (n, n)
    """
    n = len(s)
    diff = s - s0

    if issparse(Q):
        Q_diff = Q.dot(diff)
        # For sparse Q, construct sparse Jacobian
        from scipy.sparse import diags

        # Diagonal part: 3*s_k^2*(Q(s-s0))_k
        diag_part = diags(3 * s**2 * Q_diff, 0, shape=(n, n))

        # Off-diagonal part: s_k^3*Q_kj (broadcast s^3 along rows)
        s_cubed_diag = diags(s**3, 0, shape=(n, n))
        J = diag_part + s_cubed_diag.dot(Q)

        return J.toarray()  # root() expects dense Jacobian
    else:
        Q_diff = Q @ diff
        J = np.zeros((n, n))

        for k in range(n):
            J[k, k] = 3 * s[k]**2 * Q_diff[k]
            J[k, :] += s[k]**3 * Q[k, :]

        return J


def solve_nonlinear_system(Q, s0, d_tilde, initial_guess=None, method='hybr',
                           use_jacobian=True, tol=1e-8):
    """
    Solve: s_k^3 * (Q(s - s0))_k - d_tilde_k^2 = 0 for all k

    Parameters:
    -----------
    Q : sparse or dense matrix, shape (n, n)
        System matrix (typically sparse CSC format)
    s0 : array, shape (n,)
        Reference vector
    d_tilde : array, shape (n,)
        Target vector
    initial_guess : array, shape (n,), optional
        Initial guess. If None, uses s0
    method : str, optional
        Solver method: 'hybr' (default), 'lm', 'broyden1', etc.
    use_jacobian : bool, optional
        Whether to provide analytical Jacobian (faster but more memory)
    tol : float, optional
        Tolerance for convergence

    Returns:
    --------
    solution : OptimizeResult
        Contains solution.x (solution vector) and convergence info
    """
    if initial_guess is None:
        initial_guess = s0.copy()

    jac = jacobian_function if use_jacobian else None

    solution = root(
        residual_function,
        initial_guess,
        args=(Q, s0, d_tilde),
        method=method,
        jac=jac,
        tol=tol
    )

    return solution

In [ ]:
solution = solve_nonlinear_system(S_inv, s0, d_tilde, initial_guess=s0, method="broyden1", use_jacobian=False)

# Correlated prior: Diagonal fourier space kernel

In [213]:
def sample_real_field_from_power(power_spec):
    """Return a real-space Gaussian field with power spectrum `power_spec`.
    `power_spec` is the 1D power for every fft-bin (same ordering as np.fft.fftfreq).
    """
    N = len(power_spec)
    a = np.zeros(N, dtype=complex)

    # DC (real)
    a[0] = np.random.normal(scale=np.sqrt(power_spec[0]))

    # positive freq indices (exclude DC, include Nyquist if even)
    pos_end = N//2
    pos = np.arange(1, pos_end + 1) if N%2==0 else np.arange(1, pos_end+1)

    # sample complex Gaussian for positive freqs, scale so that E[|a_k|^2]=power_spec[k]
    # variance split into two for real+imag => multiply sqrt(power/2)
    re = np.random.normal(size=pos.size)
    im = np.random.normal(size=pos.size)
    a[pos] = (re + 1j*im) * np.sqrt(power_spec[pos] / 2.0)

    # impose conjugate symmetry for negative freqs
    a[-pos] = np.conj(a[pos])

    # Nyquist for even N must be real
    if N % 2 == 0:
        a[N//2] = np.random.normal(scale=np.sqrt(power_spec[N//2]))

    # real-space field
    return np.fft.ifft(a, norm="ortho").real

In [324]:
# s0 = np.sqrt(mean_prior_power_spectrum(k_lengths, (-4, 2e4)))
s0_hyper = np.sqrt(mean_prior_power_spectrum(k_lengths_unique, (-4, 1e5)))
# s0 = np.exp(data_model(s0_hyper, mode="even"))

data_samples = []
for _ in range(10):
    sl = data_model(s0, custom_norm_for_your_convenience=15)
    data_samples.append(sl)


amp_spec_samples = []
for _ in range(5):
    sl = np.exp(data_model(s0_hyper, mode="even"))*1e-4
    amp_spec_samples.append(sl)

s0 = np.mean(amp_spec_samples, axis=0)

fig, axs = plt.subplots(1,2)

# Plot amplitude spectrum
for sl in amp_spec_samples:
    axs[0].plot(k_lengths[1:], (sl**2)[1:], "-", markersize=3, )#alpha=0.1, color="black")

axs[0].plot(k_lengths[1:], (s0**2)[1:], ".", markersize=3, label="Mean prior power spectrum")
axs[0].loglog()

for sl in data_samples:
    axs[1].plot(t, sl, alpha=0.1, color="black")


axs[1].plot(t, data_samples[0], label="Single data realization")
axs[0].legend()
axs[1].legend()
axs[0].set_title("Prior power spectra")
axs[1].set_title("Data realizations from mean prior power spectrum")




k here:  [0.0000000e+00 5.0006104e-01 1.0001221e+00 ... 2.0467498e+03
 2.0472499e+03 2.0477500e+03]
step 2:  [1.0000000e+00 5.0006104e-01 1.0001221e+00 ... 2.0467498e+03
 2.0472499e+03 2.0477500e+03]
step 3:  [1.0000000e+00 1.5992189e+01 9.9951184e-01 ... 5.6982434e-14
 5.6926774e-14 5.6871181e-14]
any infs in step3?  18.30872
step 4:  [1.0000000e-30 3.4340997e+05 2.1463123e+04 ... 1.2236183e-09
 1.2224231e-09 1.2212293e-09]
(note: was divided by  99999.984


Text(0.5, 1.0, 'Data realizations from mean prior power spectrum')

In [351]:
power_spec_samples = []
for _ in range(5000):
    xi = np.random.standard_normal(len(d_tilde)) + 1j*np.random.standard_normal(len(d_tilde))
    sl = np.abs(d_tilde)**2 / np.abs(xi)**2
    power_spec_samples.append(sl)
    if np.any(np.isnan(sl)):
        print("NAN detected")
    if np.any(sl==0):
        print("zero detected")

mean_power_spec = np.mean(power_spec_samples, axis=0)


In [349]:
# for sl in amp_spec_samples:
#     plt.plot(k_lengths[1:], (sl**2)[1:], ".", markersize=3, alpha=0.1)

plt.plot(k_lengths[1:], (mean_power_spec[0]**2)[1:], ".", markersize=3, label="One realization")
plt.plot(k_lengths[1:], (mean_power_spec**2)[1:], ".", markersize=3, label="Mean power spectrum")
plt.legend()
plt.loglog()
plt.show()

In [353]:
data_realization = data_model(np.sqrt(mean_power_spec))
plt.plot(t,data_realization, label="Data realization")
plt.plot(t,d, label="Actual data")
plt.legend()

In [354]:
posterior_xi = d_tilde/np.sqrt(mean_power_spec)

In [355]:
plt.plot(k_lengths, posterior_xi, label="Posterior distribution")

/Users/iason/PycharmProjects/stability-of-submoons/.venv/lib/python3.12/site-packages/matplotlib/cbook/__init__.py:1345: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)


In [356]:
real_space_posterior_xi = np.fft.fft(posterior_xi, norm="ortho")
_ = plt.figure()
plt.plot(t, real_space_posterior_xi, "b.", markersize=2)
import nifty8 as ift


In [357]:

xi_field = ift.Field(domain=ift.DomainTuple.make(ift.RGSpace(shape=(len(t)), distances=dt), ),
                     val=real_space_posterior_xi)


In [358]:
stress_mat, time_dual, freq_dual = Stress(xi_field)
visualize_stress(stress_mat, rows=freq_dual, cols=time_dual+t[0])


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (1.704417419364662e-18) 
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
